## *Fast PEFT with LoRA – Hands-on Tutorial (Colab CPU Friendly)*

## **Installation**

In [2]:
# 1. Install dependencies
# %pip install -q transformers==4.41.0 peft==0.11.0 datasets==2.20.0 accelerate==0.30.0 langchain langchain-community

import torch
from datasets import load_dataset
from transformers import AutoTokenizer, AutoModelForCausalLM, Trainer, TrainingArguments
from peft import LoraConfig, get_peft_model, TaskType

## **Load Dataset**

In [3]:
# 2. Load Small Dataset Subset
dataset = load_dataset("imdb", split="train[:200]")  # Only 200 samples for speed

def preprocess(example):
    return {
        "text": "Review: " + example["text"][:300] + "\nSentiment:",  # truncate
        "label": example["label"]
    }

dataset = dataset.map(preprocess)
print(f"✅ Dataset ready with {len(dataset)} examples")

Map: 100%|██████████| 200/200 [00:00<00:00, 2898.51 examples/s]

✅ Dataset ready with 200 examples


## **Load Base Model**

In [4]:
# 3. Load Base Model
model_name = "gpt2"

tokenizer = AutoTokenizer.from_pretrained(model_name)
tokenizer.pad_token = tokenizer.eos_token

model = AutoModelForCausalLM.from_pretrained(model_name)
print("✅ Model loaded!")

C:\Users\ashwi\AppData\Local\Packages\PythonSoftwareFoundation.Python.3.12_qbz5n2kfra8p0\LocalCache\local-packages\Python312\site-packages\huggingface_hub\file_download.py:143: UserWarning: `huggingface_hub` cache-system uses symlinks by default to efficiently store duplicated files but your machine does not support them in C:\Users\ashwi\.cache\huggingface\hub\models--gpt2. Caching files will still work but in a degraded version that might require more space on your disk. This warning can be disabled by setting the `HF_HUB_DISABLE_SYMLINKS_WARNING` environment variable. For more details, see https://huggingface.co/docs/huggingface_hub/how-to-cache#limitations.
To support symlinks on Windows, you either need to activate Developer Mode or to run Python as an administrator. In order to activate developer mode, see this article: https://docs.microsoft.com/en-us/windows/apps/get-started/enable-your-device-for-development
  warnings.warn(message)
C:\Users\ashwi\AppData\Local\Packages\Python

✅ Model loaded!


## **Applying LORA**

In [5]:
# 4. Apply LoRA (PEFT)
lora_config = LoraConfig(
    r=8,                          # Low rank = faster
    lora_alpha=16,
    target_modules=["c_attn"],    # GPT-2 attention layers
    lora_dropout=0.1,
    bias="none",
    task_type=TaskType.CAUSAL_LM
)

model = get_peft_model(model, lora_config)
model.print_trainable_parameters()

trainable params: 294,912 || all params: 124,734,720 || trainable%: 0.2364


C:\Users\ashwi\AppData\Local\Packages\PythonSoftwareFoundation.Python.3.12_qbz5n2kfra8p0\LocalCache\local-packages\Python312\site-packages\peft\tuners\lora\layer.py:1119: UserWarning: fan_in_fan_out is set to False but the target module is `Conv1D`. Setting fan_in_fan_out to True.
  warnings.warn(


## **Tokenization**

In [7]:
# 5. Tokenization
def tokenize(example):
    tokens = tokenizer(
        example["text"],
        truncation=True,
        padding="max_length",
        max_length=128
    )
    tokens["labels"] = tokens["input_ids"].copy()
    return tokens

tokenized_dataset = dataset.map(tokenize, batched=True, remove_columns=dataset.column_names)

Map: 100%|██████████| 200/200 [00:00<00:00, 1503.76 examples/s]


## **Training Stage**

In [8]:
# 6. Training (Fast on CPU) ( 5 - 8 mins to run )
training_args = TrainingArguments(
    output_dir="./lora_finetuned",
    per_device_train_batch_size=4,     # Small batch for CPU
    num_train_epochs=1,
    logging_steps=10,
    save_strategy="no",
    report_to="none",
    fp16=False
)

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=tokenized_dataset,
)

print("🚀 Starting training (should take 3-8 mins on CPU)...")
trainer.train()

🚀 Starting training (should take 3-8 mins on CPU)...


  0%|          | 0/50 [00:00<?, ?it/s]C:\Users\ashwi\AppData\Local\Packages\PythonSoftwareFoundation.Python.3.12_qbz5n2kfra8p0\LocalCache\local-packages\Python312\site-packages\torch\utils\data\dataloader.py:752: UserWarning: 'pin_memory' argument is set as true but no accelerator is found, then device pinned memory won't be used.
  super().__init__(loader)
 20%|██        | 10/50 [00:42<02:45,  4.15s/it]

{'loss': 6.4461, 'grad_norm': 2.047130584716797, 'learning_rate': 4e-05, 'epoch': 0.2}


 40%|████      | 20/50 [01:24<02:12,  4.41s/it]

{'loss': 6.2019, 'grad_norm': 2.02284836769104, 'learning_rate': 3e-05, 'epoch': 0.4}


...

 80%|████████  | 40/50 [02:49<00:42,  4.27s/it]

{'loss': 6.0564, 'grad_norm': 2.8563122749328613, 'learning_rate': 1e-05, 'epoch': 0.8}


100%|██████████| 50/50 [03:31<00:00,  4.23s/it]

{'loss': 5.9218, 'grad_norm': 2.7294044494628906, 'learning_rate': 0.0, 'epoch': 1.0}
{'train_runtime': 211.3814, 'train_samples_per_second': 0.946, 'train_steps_per_second': 0.237, 'train_loss': 6.134425354003906, 'epoch': 1.0}


TrainOutput(global_step=50, training_loss=6.134425354003906, metrics={'train_runtime': 211.3814, 'train_samples_per_second': 0.946, 'train_steps_per_second': 0.237, 'total_flos': 13109900083200.0, 'train_loss': 6.134425354003906, 'epoch': 1.0})

## **Save the model**

In [9]:
# 7. Save Model
model.save_pretrained("lora_model")
tokenizer.save_pretrained("lora_model")
print("✅ Model saved!")

✅ Model saved!


## **Testing Stage**

In [10]:
# 8. Improved Inference (Fixed Warning + Better Generation)
from peft import PeftModel

# Load the fine-tuned LoRA model
base_model = AutoModelForCausalLM.from_pretrained(model_name)
model = PeftModel.from_pretrained(base_model, "lora_model")
model.eval()

# Better generation parameters
input_text = "Review: This movie was absolutely fantastic!\nSentiment:"

inputs = tokenizer(input_text, return_tensors="pt")

output = model.generate(
    **inputs,
    max_new_tokens=30,
    do_sample=True,           # Important for temperature
    temperature=0.7,
    top_p=0.9,
    top_k=50,
    repetition_penalty=1.1,
    pad_token_id=tokenizer.eos_token_id,
    eos_token_id=tokenizer.eos_token_id
)

print(tokenizer.decode(output[0], skip_special_tokens=True))

Review: This movie was absolutely fantastic!
Sentiment: 4/5, I enjoyed the first time around. The film is really fun and has a great cast of characters that you can interact with throughout the


## **Examples**

In [11]:
def generate_sentiment(review):
    prompt = f"Review: {review}\nSentiment:"
    inputs = tokenizer(prompt, return_tensors="pt")

    output = model.generate(
        **inputs,
        max_new_tokens=25,
        do_sample=True,
        temperature=0.75,
        top_p=0.85,
        repetition_penalty=1.2,
        pad_token_id=tokenizer.eos_token_id
    )
    return tokenizer.decode(output[0], skip_special_tokens=True)

# Test examples
tests = [
    "This movie was absolutely fantastic!",
    "The worst film I have seen this year.",
    "Acting was okay but the story was boring."
]

for test in tests:
    print(generate_sentiment(test))
    print("-" * 60)

Review: This movie was absolutely fantastic!
Sentiment: Very Positive, Love it. I've been watching this film since the beginning and still haven't gotten around to buying a new
------------------------------------------------------------
Review: The worst film I have seen this year.
Sentiment: This is a really interesting movie, and it has some very unique aspects to its story that would not be possible without the original
------------------------------------------------------------
Review: Acting was okay but the story was boring.
Sentiment: 5/10, 3 stars! (2)


------------------------------------------------------------


## **Simple Implementation Using langchain**


In [13]:
# 9. Use with LangChain
from transformers import pipeline
from langchain_community.llms import HuggingFacePipeline

pipe = pipeline(
    "text-generation",
    model=model_name,   # You can also load the fine-tuned one
    tokenizer=tokenizer,
    max_new_tokens=50,
    pad_token_id=tokenizer.eos_token_id
)

llm = HuggingFacePipeline(pipeline=pipe)

response = llm.invoke("Review: The acting was brilliant!\nSentiment:")
print("-----------------------------------")
print(response)

ModuleNotFoundError: No module named 'langchain_community'